# Notebook 3 — Baseline Modelling

**Input:** `features_train.csv`, `features_test.csv` (from Notebook 2)

**Goal:** Establish baseline CV AUC with XGBoost, compare LightGBM & CatBoost, try simple ensemble.

**Output:** CV AUC scores, feature importance ranking.

In [1]:
import numpy as np
import pandas as pd
from scipy.stats import rankdata
from sklearn.model_selection import StratifiedKFold, cross_val_predict
from sklearn.metrics import roc_auc_score
from xgboost import XGBClassifier
from lightgbm import LGBMClassifier
from catboost import CatBoostClassifier

# ============================================================
# Load engineered features
# ============================================================
train = pd.read_csv('features_train.csv')
test  = pd.read_csv('features_test.csv')

feature_cols = [c for c in train.columns
                if c not in ['uid', 'TARGET', 'NAME_CONTRACT_TYPE']]
for c in feature_cols:
    if c not in test.columns: test[c] = 0
test = test[['uid'] + feature_cols]

X = train[feature_cols]
y = train['TARGET']
spw = float((y == 0).sum() / (y == 1).sum())
print(f'Features: {len(feature_cols)}  |  scale_pos_weight: {spw:.2f}')
print(f'Train: {X.shape}  |  Default rate: {y.mean():.4f}')

Features: 69  |  scale_pos_weight: 11.41
Train: (261383, 69)  |  Default rate: 0.0806


In [2]:
cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)

## 1. XGBoost Baseline

In [3]:
xgb_base = XGBClassifier(
    n_estimators=400, max_depth=4, learning_rate=0.05,
    subsample=0.8, colsample_bytree=0.8,
    scale_pos_weight=spw, eval_metric='auc',
    n_jobs=-1, random_state=42,
)

oof_xgb_base = cross_val_predict(xgb_base, X, y, cv=cv,
                                  method='predict_proba', n_jobs=-1)[:, 1]
auc_xgb_base = roc_auc_score(y, oof_xgb_base)
print(f'XGBoost Baseline CV AUC: {auc_xgb_base:.5f}')

XGBoost Baseline CV AUC: 0.67718


In [4]:
# Feature importance
xgb_base.fit(X, y)
imp = pd.Series(xgb_base.feature_importances_, index=feature_cols).sort_values(ascending=False)
print('TOP 20 FEATURES:')
print(imp.head(20))

TOP 20 FEATURES:
acc_days_since_open_mean    0.075839
enq_days_since_first        0.053374
is_cash_loan                0.051230
acc_n_open                  0.050572
enq_n_180d                  0.036038
enq_n_90d                   0.034823
acc_cnt_Microloan_share     0.034717
acc_open_ratio              0.031835
acc_days_since_open_min     0.028335
acc_overdue_sum             0.022503
acc_overdue_max             0.020916
pmt_recent_dpd_max          0.019506
acc_cnt_Microloan           0.018188
acc_loan_amt_mean           0.016833
enq_span_days               0.016749
enq_n_365d                  0.016532
enq_log_amt_sum             0.016453
acc_days_since_open_max     0.015665
acc_overdue_amt_ratio       0.015008
enq_days_since_last         0.014613
dtype: float32


## 2. LightGBM

In [5]:
lgb = LGBMClassifier(
    n_estimators=500, max_depth=3, num_leaves=8,
    learning_rate=0.05, subsample=0.75, colsample_bytree=0.65,
    reg_alpha=3.5, reg_lambda=8.5, min_child_samples=40,
    scale_pos_weight=spw, random_state=42, n_jobs=-1, verbose=-1
)
oof_lgb = cross_val_predict(lgb, X, y, cv=cv, method='predict_proba', n_jobs=-1)[:, 1]
print(f'LightGBM CV AUC: {roc_auc_score(y, oof_lgb):.5f}')

LightGBM CV AUC: 0.67810


## 3. CatBoost

In [6]:
cat = CatBoostClassifier(
    iterations=500, depth=3, learning_rate=0.05,
    l2_leaf_reg=8, subsample=0.75,
    scale_pos_weight=spw, random_state=42,
    verbose=0, allow_writing_files=False
)
oof_cat = cross_val_predict(cat, X, y, cv=cv, method='predict_proba', n_jobs=1)[:, 1]
print(f'CatBoost CV AUC: {roc_auc_score(y, oof_cat):.5f}')

CatBoost CV AUC: 0.67752


## 4. Ensemble

In [7]:
print('Model correlations (lower = more diverse):')
print(f'  xgb-lgb: {np.corrcoef(oof_xgb_base, oof_lgb)[0,1]:.4f}')
print(f'  xgb-cat: {np.corrcoef(oof_xgb_base, oof_cat)[0,1]:.4f}')
print(f'  lgb-cat: {np.corrcoef(oof_lgb, oof_cat)[0,1]:.4f}')

blend = (oof_xgb_base + oof_lgb + oof_cat) / 3
rank_blend = (rankdata(oof_xgb_base) + rankdata(oof_lgb) + rankdata(oof_cat)) / 3

print(f'\nBlend      CV AUC: {roc_auc_score(y, blend):.5f}')
print(f'Rank-Blend CV AUC: {roc_auc_score(y, rank_blend):.5f}')

print('\n========== SUMMARY ==========')
for name, oof in [('XGBoost', oof_xgb_base), ('LightGBM', oof_lgb),
                  ('CatBoost', oof_cat), ('Blend', blend),
                  ('Rank-Blend', rank_blend)]:
    print(f'  {name:12s}: {roc_auc_score(y, oof):.5f}')

Model correlations (lower = more diverse):
  xgb-lgb: 0.9834
  xgb-cat: 0.9740
  lgb-cat: 0.9850

Blend      CV AUC: 0.67860
Rank-Blend CV AUC: 0.67857

========== SUMMARY ==========
  XGBoost     : 0.67718
  LightGBM    : 0.67810
  CatBoost    : 0.67752
  Blend       : 0.67860
  Rank-Blend  : 0.67857
